# DWPose body reconstruction + SMPL-X fit

Full DWPose pipeline, parallel to `body_keypoint_reconstruction.ipynb` /
`smplx_body_fit.ipynb`:

1. Extract DWPose 18-joint body keypoints (YOLOX + RTMPose ONNX) from the H5
   recording and cache to `dwpose_keypoints.npz`.
2. RANSAC multi-view triangulation — same calibration, thresholds, and Kalman/SG
   smoothing pipeline as the MediaPipe body notebook.
3. SMPL-X body-only fit (`v3.smplx_body_fit`) to **all** valid frames using
   `DWPOSE_JOINT_CORRESPONDENCES` and warm-starting from the previous frame.
4. Animated viser viewer: SMPL-X mesh + DWPose 3D skeleton, looping playback
   with play/pause button and frame-slider transport.

Generated by `v3/notebooks/build_dwpose_notebook.py` — edit that script and
re-run `python -m v3.notebooks.build_dwpose_notebook`, do not edit this
notebook directly.  Runs in `oak_env`.

In [ ]:
import os
import time

import numpy as np
import torch

from v3.calibration import load_calibration
from v3.dwpose_extraction import load_dwpose, extract_keypoints, N_BODY
from v3.triangulation import undistort_keypoints, ransac_triangulate_sequence
from v3.smoothing import smooth_sequence, jitter_flags, savgol_and_upsample_2x
from v3.smplx_body_fit import (build_model, get_targets, fit_frame,
                                DWPOSE_JOINT_CORRESPONDENCES)
from v3.skeletons import DWPOSE_BODY_CONNECTIONS, SMPLX_BODY_BONES
from v3 import visualization as viz

H5_PATH = '/data/fmalmb/DATA/tmp/Testdata.h5'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
CAMERA_CALIB, CAMERA_WORLD_TRANSFORMS, PROJECTION_MATRICES, camera_ids = load_calibration(H5_PATH)
N_CAMS = len(camera_ids)

model = build_model(DEVICE)
print(f'SMPL-X model loaded on {DEVICE}')
print(f'Cameras: {camera_ids}')

In [ ]:
# --- Stage 1: extract DWPose keypoints (or load from cache) --------------------------
FRAME_OFFSET = 0
N_FRAMES = 128
frame_keys = [f'{i:06d}' for i in range(FRAME_OFFSET, FRAME_OFFSET + N_FRAMES)]

NPZ_CACHE = 'dwpose_keypoints.npz'

if os.path.exists(NPZ_CACHE):
    d = np.load(NPZ_CACHE)
    keypoints_px = d['keypoints_px']
    keypoints_conf = d['keypoints_conf']
    presence = d['presence']
    print(f'Loaded DWPose keypoints from cache ({NPZ_CACHE})')
else:
    session_det, session_pose = load_dwpose(use_gpu=(DEVICE.type == 'cuda'))
    ep = session_det.get_providers()[0]
    print(f'DWPose models loaded ({ep})')
    print(f'Extracting: {N_FRAMES} frames × {N_CAMS} cameras ...')
    t0 = time.time()
    keypoints_px, keypoints_conf, presence = extract_keypoints(
        H5_PATH, camera_ids, frame_keys, session_det, session_pose, verbose=True)
    print(f'Done in {time.time() - t0:.1f}s')
    np.savez_compressed(NPZ_CACHE,
                        keypoints_px=keypoints_px,
                        keypoints_conf=keypoints_conf,
                        presence=presence)
    print(f'Saved to {NPZ_CACHE}')

print()
for ci, cam_id in enumerate(camera_ids):
    print(f'{cam_id}: {int(presence[ci].sum())}/{N_FRAMES} frames with person detected')

In [ ]:
# --- Stage 2: RANSAC multi-view triangulation ----------------------------------------
# seed_joint_idx=(8, 11): right_hip and left_hip in DWPose's OpenPose order.
# Same reproj_thresh_px=50.0 as the MediaPipe body pipeline.
RANSAC_REPROJ_THRESH_PX = 50.0

keypoints_px_undist = undistort_keypoints(keypoints_px, CAMERA_CALIB, camera_ids)

result = ransac_triangulate_sequence(
    keypoints_px_undist, keypoints_conf, presence, PROJECTION_MATRICES, camera_ids,
    seed_joint_idx=(8, 11),
    occlusion=None,
    reproj_thresh_px=RANSAC_REPROJ_THRESH_PX,
)
joints3d = result['joints3d']
valid = result['valid']
n_inliers = result['n_inliers']
reproj_err = result['reproj_err']
hip_residuals = result['seed_residuals']
hysteresis_overrides = result['hysteresis_overrides']
n_valid_frames = result['n_valid_frames']

n_valid = int(valid.sum())
ni = n_inliers[valid]
print(f'body (DWPose): {n_valid}/{N_FRAMES} frames valid')
print(f'  inlier cams: min={ni.min()}, mean={ni.mean():.2f}, max={ni.max()}')
print(f'  mean reprojection error: {np.nanmean(reproj_err[valid]):.1f}px')
print(f'  hysteresis kept previous camera set: {int(hysteresis_overrides)}/{int(n_valid_frames)} valid frames')
print(f'hip-center RANSAC residual (px): mean={hip_residuals.mean():.1f}, '
      f'median={np.median(hip_residuals):.1f}, max={hip_residuals.max():.1f}')

ORIGIN_WORLD = viz.compute_origin(joints3d, valid)

In [ ]:
# --- Stage 3: Kalman/RTS smoothing -----------------------------------------------
KALMAN_PROCESS_STD = 0.03
KALMAN_MEAS_STD_BASE = 0.01
KALMAN_MEAS_STD_2CAM = 0.015
MIN_INLIERS_BASE = 3

joints3d_smooth, valid_smooth = smooth_sequence(
    joints3d, valid, n_inliers,
    process_std=KALMAN_PROCESS_STD,
    meas_std_base=KALMAN_MEAS_STD_BASE,
    meas_std_2cam=KALMAN_MEAS_STD_2CAM,
    min_inliers_base=MIN_INLIERS_BASE,
)

hip_raw = (joints3d[:, 8] + joints3d[:, 11]) / 2.0
hip_smooth = (joints3d_smooth[:, 8] + joints3d_smooth[:, 11]) / 2.0
delta = np.linalg.norm(hip_raw[valid] - hip_smooth[valid], axis=-1)
print(f'hip-center Kalman correction (cm): '
      f'mean={delta.mean()*100:.2f}, median={np.median(delta)*100:.2f}, max={delta.max()*100:.2f}')

In [ ]:
# --- Stage 4: jitter flagging (hip-center raw vs Kalman/RTS-smoothed) ---------------
JITTER_MAD_FACTOR = 4.0

jitter_flag, hip_jitter_residual = jitter_flags(
    joints3d, joints3d_smooth, valid, seed_joint_idx=(8, 11), mad_factor=JITTER_MAD_FACTOR)

valid_idx = np.where(valid)[0]
res = hip_jitter_residual[valid_idx]
med = np.median(res)
mad = np.median(np.abs(res - med)) + 1e-9
thresh = med + JITTER_MAD_FACTOR * mad
flagged = valid_idx[res > thresh]

print(f'hip-center raw-vs-smoothed residual (cm): median={med*100:.2f}, '
      f'mad={mad*100:.2f}, threshold={thresh*100:.2f}, max={res.max()*100:.2f}')
print(f'flagged frames ({len(flagged)}/{len(valid_idx)}): {(flagged + FRAME_OFFSET).tolist()}')

In [ ]:
# --- Stage 5: Savitzky-Golay + 2x frame-rate upsample + save --------------------
SG_WINDOW = 7
SG_POLYORDER = 3

sg_result = savgol_and_upsample_2x(joints3d_smooth, valid_smooth,
                                    sg_window=SG_WINDOW, sg_polyorder=SG_POLYORDER)
joints3d_sg = sg_result['joints3d_sg']
joints3d_2x = sg_result['joints3d_2x']
valid_2x = sg_result['valid_2x']
N_FRAMES_2X = sg_result['n_frames_2x']
nearest_orig_2x = sg_result['nearest_orig_2x']

jitter_flag_2x = jitter_flag[nearest_orig_2x]
n_inliers_2x = n_inliers[nearest_orig_2x]

hip_smooth = (joints3d_smooth[:, 8] + joints3d_smooth[:, 11]) / 2.0
hip_sg = (joints3d_sg[:, 8] + joints3d_sg[:, 11]) / 2.0
delta = np.linalg.norm(hip_smooth[valid_smooth] - hip_sg[valid_smooth], axis=-1)
print(f'SG correction (cm): mean={delta.mean()*100:.3f}, max={delta.max()*100:.3f}')
print(f'{int(valid_2x.sum())}/{N_FRAMES_2X} frames valid at 2x rate')

np.savez_compressed(
    'dwpose_keypoint_3d.npz',
    joints3d=joints3d, valid=valid, n_inliers=n_inliers, reproj_err=reproj_err,
    joints3d_smooth=joints3d_smooth, valid_smooth=valid_smooth,
    joints3d_sg=joints3d_sg, joints3d_2x=joints3d_2x, valid_2x=valid_2x,
    n_inliers_2x=n_inliers_2x, jitter_flag=jitter_flag, jitter_flag_2x=jitter_flag_2x,
    nearest_orig_2x=nearest_orig_2x, n_frames_2x=np.array(N_FRAMES_2X),
    camera_ids=np.array(camera_ids), frame_indices=np.arange(FRAME_OFFSET, FRAME_OFFSET + N_FRAMES),
)
print('Saved dwpose_keypoint_3d.npz')

In [ ]:
# --- Stage 6: SMPL-X fit to ALL valid frames (warm-start from previous frame) ------
N_ITERS = 200   # fewer iterations needed with warm-start; convergence is fast
LR = 0.05
W_BETAS = 1e-2
W_POSE = 1e-3
PRINT_INTERVAL = 16  # print progress every N fitted frames

fit_results = {}
prev_params = None
n_fit = 0
t0 = time.time()

for fi in range(N_FRAMES):
    if not valid[fi]:
        continue
    target, weights, smplx_idx, labels = get_targets(joints3d_sg[fi], DWPOSE_JOINT_CORRESPONDENCES)
    result = fit_frame(
        model, target, weights, smplx_idx, DEVICE,
        n_iters=N_ITERS, lr=LR, w_betas=W_BETAS, w_pose=W_POSE,
        print_every=0, label=f'f{fi}',
        warm_start=prev_params,
    )
    result['target'] = target
    fit_results[fi] = result
    prev_params = {k: result[k].copy() for k in ('global_orient', 'transl', 'body_pose', 'betas')}
    n_fit += 1
    if n_fit == 1 or n_fit % PRINT_INTERVAL == 0:
        elapsed = time.time() - t0
        print(f'  [{n_fit:3d}/{int(valid.sum())} frames]  f{fi:03d}  '
              f'loss={result["losses"][-1]:.4f}  {elapsed:.1f}s elapsed', flush=True)

elapsed = time.time() - t0
print(f'\nFitted {n_fit} frames in {elapsed:.1f}s  ({elapsed / max(n_fit, 1):.2f}s/frame)')

all_res = np.stack([fit_results[fi]['residuals'] for fi in sorted(fit_results)])
print('\nMean residuals across all fitted frames (mm):')
for lbl, r in zip(labels, all_res.mean(axis=0)):
    print(f'  {lbl:16s}: {r * 1000:.1f}mm')
print(f'  {"overall":16s}: {all_res.mean() * 1000:.1f}mm')

In [ ]:
# --- Stage 7: animated viser viewer ---------------------------------------------------
import viser

SMPLX_FACES_UINT32 = model.faces.astype(np.uint32)
MESH_COLOR = (180, 200, 230)      # light blue-grey SMPL-X mesh
SMPLX_SKEL_COLOR = (100, 220, 100)  # green SMPL-X skeleton
DWPOSE_COLOR = (255, 170, 40)     # amber DWPose 3D skeleton

# Pre-transform everything to viser space once (avoids per-frame Python loops).
print('Pre-computing viser-space geometry ...')
valid_fit_frames = sorted(fit_results.keys())
fi_to_vidx = {fi: i for i, fi in enumerate(valid_fit_frames)}

all_verts_viser = np.stack([
    ((fit_results[fi]['vertices'] - ORIGIN_WORLD) @ viz.R_WORLD_TO_VISER.T).astype(np.float32)
    for fi in valid_fit_frames
])  # (n_fit, n_verts, 3)

all_smplx_joints_viser = np.stack([
    np.stack([viz.world_to_viser_point(p, ORIGIN_WORLD)
              for p in fit_results[fi]['joints'][:22]]).astype(np.float32)
    for fi in valid_fit_frames
])  # (n_fit, 22, 3)

dwpose_pts_2x = {}
for fi2x in range(N_FRAMES_2X):
    if valid_2x[fi2x]:
        dwpose_pts_2x[fi2x] = np.stack([
            viz.world_to_viser_point(p, ORIGIN_WORLD)
            for p in joints3d_2x[fi2x]
        ]).astype(np.float32)

SMPLX_BONE_SEGS = np.array([(a, b) for a, b in SMPLX_BODY_BONES], dtype=np.int32)
DWPOSE_SEGS = np.array([(a, b) for a, b in DWPOSE_BODY_CONNECTIONS], dtype=np.int32)
print(f'Pre-computation done.  {len(valid_fit_frames)} SMPL-X fits, {len(dwpose_pts_2x)} 2x skeleton frames.')


def _empty_cloud():
    return np.zeros((0, 3), dtype=np.float32), np.zeros((0, 3), dtype=np.uint8)

def _empty_lines():
    return np.zeros((0, 2, 3), dtype=np.float32), np.zeros((0, 2, 3), dtype=np.uint8)


def main():
    server = viser.ViserServer()
    running = True
    try:
        server.scene.set_up_direction('+z')
        server.gui.configure_theme(dark_mode=True)
        viz.setup_cameras(server, CAMERA_CALIB, CAMERA_WORLD_TRANSFORMS, ORIGIN_WORLD)

        server.initial_camera.position = (2.5, 2.0, 1.6)
        server.initial_camera.look_at = (0.0, 0.0, 0.9)
        server.initial_camera.up = (0.0, 0.0, 1.0)

        print('Viser: open the URL shown above')

        stop_button = server.gui.add_button('Stop viewer')

        with server.gui.add_folder('Playback'):
            frame_slider = server.gui.add_slider(
                'Frame (2x)', min=0, max=N_FRAMES_2X - 1, step=1, initial_value=0)
            play_button = server.gui.add_button('▶ / ⏸  Play / Pause')
            fps_slider = server.gui.add_slider('FPS', min=1, max=60, step=1, initial_value=20)

        with server.gui.add_folder('Visibility'):
            show_mesh = server.gui.add_checkbox('SMPL-X mesh', initial_value=True)
            show_smplx_skel = server.gui.add_checkbox('SMPL-X skeleton', initial_value=False)
            show_dwpose = server.gui.add_checkbox('DWPose 3D skeleton', initial_value=True)

        @stop_button.on_click
        def _stop(_):
            nonlocal running
            running = False

        is_playing = [False]

        @play_button.on_click
        def _toggle(_):
            is_playing[0] = not is_playing[0]

        last_rendered_fi = [-1]

        def _force_rerender(_):
            last_rendered_fi[0] = -1

        for ctrl in (frame_slider, show_mesh, show_smplx_skel, show_dwpose):
            ctrl.on_update(_force_rerender)

        def render_frame(fi2x):
            orig_fi = int(nearest_orig_2x[fi2x])
            vidx = fi_to_vidx.get(orig_fi)

            # SMPL-X mesh
            if vidx is not None and show_mesh.value:
                server.scene.add_mesh_simple(
                    '/smplx/mesh', all_verts_viser[vidx], SMPLX_FACES_UINT32,
                    color=MESH_COLOR, opacity=0.85, flat_shading=False)
            else:
                if vidx is not None:  # hide: render with opacity 0
                    server.scene.add_mesh_simple(
                        '/smplx/mesh', all_verts_viser[vidx], SMPLX_FACES_UINT32,
                        color=MESH_COLOR, opacity=0.0, flat_shading=False)

            # SMPL-X skeleton
            if vidx is not None and show_smplx_skel.value:
                pts = all_smplx_joints_viser[vidx]
                segs = np.stack([[pts[a], pts[b]] for a, b in SMPLX_BODY_BONES]).astype(np.float32)
                server.scene.add_point_cloud(
                    '/smplx/joints', pts,
                    colors=np.full((22, 3), SMPLX_SKEL_COLOR, dtype=np.uint8),
                    point_size=0.01)
                server.scene.add_line_segments(
                    '/smplx/bones', segs,
                    colors=np.array(SMPLX_SKEL_COLOR, dtype=np.uint8), line_width=2.0)
            else:
                pts_e, col_e = _empty_cloud()
                server.scene.add_point_cloud('/smplx/joints', pts_e, colors=col_e)
                segs_e, col_e = _empty_lines()
                server.scene.add_line_segments('/smplx/bones', segs_e, colors=col_e)

            # DWPose 3D skeleton (2x SG-smoothed)
            if fi2x in dwpose_pts_2x and show_dwpose.value:
                pts = dwpose_pts_2x[fi2x]
                segs = np.stack([[pts[a], pts[b]] for a, b in DWPOSE_BODY_CONNECTIONS]).astype(np.float32)
                server.scene.add_point_cloud(
                    '/dwpose/joints', pts,
                    colors=np.full((N_BODY, 3), DWPOSE_COLOR, dtype=np.uint8),
                    point_size=0.01)
                server.scene.add_line_segments(
                    '/dwpose/bones', segs,
                    colors=np.array(DWPOSE_COLOR, dtype=np.uint8), line_width=2.0)
            else:
                pts_e, col_e = _empty_cloud()
                server.scene.add_point_cloud('/dwpose/joints', pts_e, colors=col_e)
                segs_e, col_e = _empty_lines()
                server.scene.add_line_segments('/dwpose/bones', segs_e, colors=col_e)

        render_frame(0)
        last_rendered_fi[0] = 0

        last_advance = time.time()

        while running:
            fi = frame_slider.value
            if fi != last_rendered_fi[0]:
                render_frame(fi)
                last_rendered_fi[0] = fi

            if is_playing[0]:
                now = time.time()
                if now - last_advance >= 1.0 / fps_slider.value:
                    frame_slider.value = (fi + 1) % N_FRAMES_2X
                    last_advance = now

            time.sleep(0.01)

    except KeyboardInterrupt:
        print('Viewer interrupted.')
    finally:
        server.stop()
        print('Viser stopped.')


main()